# IPSA (CC01-1940): modelos de clasificación

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juanestebancg2806/sugarcane-yield-quality-prediction/blob/main/02_modelos_clasificacion.ipynb)

Este notebook entrena y evalúa los modelos de clasificación de la parte de `BD_IPSA_1940` del taller.
Continúa el análisis de `01_eda_ipsa.ipynb`, donde se definieron los predictores, el tratamiento de los
valores faltantes y las clases.

**Objetivo:** clasificar cada cosecha en un nivel bajo, medio o alto de TCH y de sacarosa, como dos
problemas independientes, con información disponible antes de la cosecha. Se comparan dos modelos
(regresión logística multinomial con regularización y KNN) y dos definiciones de clase (umbrales del
Ingenio y terciles).

**Entrada.** El archivo `datos/ipsa_limpio.parquet`, generado al final de `01_eda_ipsa.ipynb`. Contiene
los targets, la marca `excluir_tch`, los 13 predictores seleccionados y las clases definidas con los
umbrales del Ingenio. En Colab hay que ejecutar primero `01_eda_ipsa.ipynb` en la misma sesión, o subir
el archivo `ipsa_limpio.parquet` a la carpeta `datos/`.

**Criterios que vienen del EDA**

- La partición en entrenamiento (80 %) y prueba (20 %) mantiene juntas las cosechas de una misma suerte,
  porque 629 suertes aparecen en más de una cosecha, y conserva la proporción de cada clase.
- Todos los estadísticos se calculan solo con el conjunto de entrenamiento. La imputación, el acotamiento
  y el escalado se ajustan dentro de un pipeline, de modo que se recalculan en cada pliegue de la
  validación cruzada. Los cortes de los terciles se calculan una vez, sobre el conjunto de entrenamiento.
- Los 4 registros marcados con `excluir_tch` se excluyen de la clasificación de TCH y se conservan en la
  de sacarosa.
- Las clases no están balanceadas con los umbrales del Ingenio, así que se reportan métricas por clase
  (precisión, sensibilidad y F1), el F1 macro y el índice kappa, además de la exactitud. La sensibilidad de
  la clase baja tiene especial importancia, porque corresponde a las suertes que el Ingenio necesita
  detectar.

**Flujo del notebook**

10. Carga del dataset procesado
11. Partición en entrenamiento y prueba
12. Clases por terciles
13. Pipeline de preprocesamiento
14. Regresión logística multinomial
15. KNN
16. Evaluación de variables candidatas (`anio`, armónicos del mes, `lluvia_sin_dato`, `edad²` y un
    conjunto reducido para KNN)
17. Evaluación en el conjunto de prueba
18. Interpretación y conclusiones

> **Datos:** `datos/ipsa_limpio.parquet` no está en el repositorio porque el enunciado prohíbe publicar
> los datos.

In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", None)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11

%matplotlib inline
print("scikit-learn:", sklearn.__version__)

scikit-learn: 1.9.1


## 10. Carga del dataset procesado

Se carga el dataset generado por `01_eda_ipsa.ipynb` y se verifica que conserva las propiedades
establecidas en el EDA: número de registros y columnas, registros marcados con `excluir_tch`, clases
del Ingenio y valores faltantes en los predictores. Si alguna verificación falla, el archivo no
corresponde a la versión final del EDA y el notebook se detiene.

In [4]:
# 10. Carga del dataset procesado
RANDOM_STATE = 42  # mismo valor que en el análisis de regresión del histórico de suertes

df = pd.read_parquet("datos/ipsa_limpio.parquet")

# Predictores definidos en la sección 7 del EDA
predictores_num = ["edad", "cortes", "semsmad", "lluvias", "pct_diatrea", "dosismad", "anio"]
predictores_bin = ["lluvia_sin_dato"]
predictores_mes = ["mes_sin_12", "mes_cos_12", "mes_sin_6", "mes_cos_6"]
predictores_cat = ["grupo_tenencia"]
predictores = predictores_num + predictores_bin + predictores_mes + predictores_cat

# Verificaciones de lo establecido en el EDA
assert df.shape == (2187, 22), f"Dimensiones inesperadas: {df.shape}"
assert set(predictores) <= set(df.columns), "Faltan predictores en el archivo"
assert df["excluir_tch"].sum() == 4, "El número de registros con excluir_tch no es 4"
assert df.loc[df["excluir_tch"], "clase_tch_ingenio"].isna().all(), "Registros excluidos con clase de TCH"
assert df.loc[~df["excluir_tch"], "clase_tch_ingenio"].notna().all(), "Registros válidos sin clase de TCH"
assert df["clase_sac_ingenio"].notna().all(), "Registros sin clase de sacarosa"
assert df["clase_tch_ingenio"].cat.ordered, "Las clases perdieron su orden (bajo < medio < alto)"

nulos_esperados = {"lluvias": 685, "semsmad": 2, "dosismad": 2}
nulos = df[predictores].isna().sum()
assert nulos[nulos > 0].to_dict() == nulos_esperados, f"Faltantes inesperados: {nulos[nulos > 0].to_dict()}"

print("Registros:", len(df), "| Predictores:", len(predictores))
display(df.head())

Registros: 2187 | Predictores: 13


,FAZ,TAL,periodo,mes,TCH,sacarosa,excluir_tch,edad,cortes,semsmad,lluvias,pct_diatrea,dosismad,anio,lluvia_sin_dato,mes_sin_12,mes_cos_12,mes_sin_6,mes_cos_6,grupo_tenencia,clase_tch_ingenio,clase_sac_ingenio
0,81291,40,202012,12,112,14.0,False,12.3,4,8.3,137.0,6.2,0.8,2020,False,-2.449294e-16,1.000000e+00,-4.898587e-16,1.0,3,bajo,alto
1,81291,41,201903,3,157,13.0,False,11.2,2,6.3,NaN,3.5,0.8,2019,True,1.000000e+00,6.123234e-17,1.224647e-16,-1.0,3,alto,alto
2,81291,41,202003,3,167,13.3,False,12.2,3,7.9,68.0,4.3,0.6,2020,False,1.000000e+00,6.123234e-17,1.224647e-16,-1.0,3,alto,alto
3,81291,43,201903,3,156,13.4,False,13.1,1,6.6,NaN,3.5,0.8,2019,True,1.000000e+00,6.123234e-17,1.224647e-16,-1.0,3,alto,alto
4,81291,43,202003,3,151,14.0,False,12.2,2,8.1,68.0,4.3,0.6,2020,False,1.000000e+00,6.123234e-17,1.224647e-16,-1.0,3,alto,alto


### Conclusión de la sección 10

El dataset tiene 2.187 registros y 22 columnas, y cumple todas las propiedades establecidas en el EDA:
13 predictores, 4 registros marcados con `excluir_tch` sin clase de TCH, clases de sacarosa completas con
el orden bajo < medio < alto, y faltantes solo en `lluvias` (685), `semsmad` (2) y `dosismad` (2).

## 11. Partición en entrenamiento y prueba

Los datos se dividen en entrenamiento (80 %) y prueba (20 %) antes de cualquier imputación, escalado o
cálculo de terciles. El conjunto de prueba no se vuelve a usar hasta la evaluación final (sección 17).

La partición cumple tres condiciones:

- **Agrupada por suerte.** Todas las cosechas de una misma suerte quedan en el mismo conjunto. Así, el
  desempeño en prueba mide la capacidad del modelo para clasificar suertes que no vio en el
  entrenamiento, y no su capacidad para recordar el historial de una suerte conocida. En el análisis de
  regresión del histórico de suertes se usó una partición aleatoria de registros y se documentó esta
  limitación, señalando la partición por suerte como la alternativa adecuada.
- **Estratificada.** Conserva en ambos conjuntos la proporción de cada combinación de clases de TCH y de
  sacarosa (umbrales del Ingenio). Los cuatro registros con `excluir_tch` tienen TCH menor que 60 t/ha,
  así que para la estratificación se agrupan con la clase baja de TCH.
- **Común a ambos targets.** Los dos problemas se evalúan sobre las mismas suertes de prueba.

Se usa `StratifiedGroupKFold` con 5 pliegues y se toma uno de ellos como conjunto de prueba, lo que
equivale a una proporción de aproximadamente 20 %.

In [5]:
# 11. Partición en entrenamiento y prueba: agrupada por suerte y estratificada
from sklearn.model_selection import StratifiedGroupKFold

# Identificador de suerte (hacienda + suerte), usado para agrupar
df["suerte"] = df["FAZ"].astype(str) + "_" + df["TAL"]

# Estrato: combinación de las clases del Ingenio de ambos targets
# Los 4 registros con excluir_tch tienen TCH < 60, así que solo para estratificar se agrupan con "bajo"
estrato = (df["clase_tch_ingenio"].astype(str).where(~df["excluir_tch"], "bajo")
           + "|" + df["clase_sac_ingenio"].astype(str))

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
idx_train, idx_test = next(sgkf.split(df, estrato, groups=df["suerte"]))
df_train, df_test = df.iloc[idx_train].copy(), df.iloc[idx_test].copy()

# Ninguna suerte puede estar en ambos conjuntos
assert set(df_train["suerte"]).isdisjoint(df_test["suerte"]), "Hay suertes en train y en test"

print(f"Train: {len(df_train)} registros ({len(df_train) / len(df):.1%}) | "
      f"{df_train['suerte'].nunique()} suertes")
print(f"Test:  {len(df_test)} registros ({len(df_test) / len(df):.1%}) | "
      f"{df_test['suerte'].nunique()} suertes")
print("Registros con excluir_tch en train / test:",
      df_train["excluir_tch"].sum(), "/", df_test["excluir_tch"].sum())

# Proporción de cada clase en train y en test, por target
balance_split = pd.DataFrame({
    ("TCH", "train"): df_train["clase_tch_ingenio"].value_counts(normalize=True),
    ("TCH", "test"): df_test["clase_tch_ingenio"].value_counts(normalize=True),
    ("sacarosa", "train"): df_train["clase_sac_ingenio"].value_counts(normalize=True),
    ("sacarosa", "test"): df_test["clase_sac_ingenio"].value_counts(normalize=True),
}).mul(100).round(1)
display(balance_split)

Train: 1749 registros (80.0%) | 892 suertes
Test:  438 registros (20.0%) | 223 suertes
Registros con excluir_tch en train / test: 4 / 0


TCH       sacarosa      
      train  test    train  test
bajo   22.6  22.8     21.6  21.5
medio  39.0  38.8     36.7  36.5
alto   38.5  38.4     41.7  42.0

### Conclusión de la sección 11

| | Registros | Suertes |
|---|---|---|
| Entrenamiento | 1.749 (80.0 %) | 892 |
| Prueba | 438 (20.0 %) | 223 |

- Ninguna suerte aparece en ambos conjuntos, así que el desempeño en prueba corresponde a suertes que el
  modelo no vio durante el entrenamiento.
- La proporción de cada clase del Ingenio difiere como máximo 0.3 puntos porcentuales entre
  entrenamiento y prueba, en ambos targets:

| Clase | TCH train | TCH test | Sacarosa train | Sacarosa test |
|---|---|---|---|---|
| Bajo | 22.6 % | 22.8 % | 21.6 % | 21.5 % |
| Medio | 39.0 % | 38.8 % | 36.7 % | 36.5 % |
| Alto | 38.5 % | 38.4 % | 41.7 % | 42.0 % |

- Los cuatro registros con `excluir_tch` quedaron en el conjunto de entrenamiento. La clasificación de
  TCH usa 1.745 registros de entrenamiento y los 438 de prueba; la de sacarosa usa los 1.749 y los 438.

## 12. Clases por terciles

La segunda definición de clase divide cada target en tres grupos del mismo tamaño. Los cortes (percentiles
33 y 67) se calculan solo con el conjunto de entrenamiento y se aplican a ambos conjuntos, de modo que las
clases del conjunto de prueba se asignan con fronteras calculadas sin sus datos. En TCH se excluyen del
cálculo los registros con `excluir_tch`, que tampoco reciben clase.

Se comparan los cortes obtenidos con los terciles de referencia del EDA, calculados sobre el dataset
completo, y con los umbrales del Ingenio, y se verifica el balance de las clases en ambos conjuntos.

In [6]:
# 12. Clases por terciles, con cortes calculados solo en el conjunto de entrenamiento
orden_clases = pd.CategoricalDtype(["bajo", "medio", "alto"], ordered=True)

config_terciles = {
    "TCH": {"col_clase": "clase_tch_terciles", "validos": ~df_train["excluir_tch"]},
    "sacarosa": {"col_clase": "clase_sac_terciles", "validos": df_train["sacarosa"].notna()},
}

cortes_terciles = {}
for tgt, cfg in config_terciles.items():
    p33, p67 = df_train.loc[cfg["validos"], tgt].quantile([1 / 3, 2 / 3])
    cortes_terciles[tgt] = (p33, p67)
    # Mismo criterio que pd.qcut: cada intervalo incluye su límite superior
    for d in (df_train, df_test):
        d[cfg["col_clase"]] = pd.cut(d[tgt], bins=[-np.inf, p33, p67, np.inf],
                                     labels=orden_clases.categories).astype(orden_clases)

# Los registros con excluir_tch no tienen clase de TCH en ninguna definición
for d in (df_train, df_test):
    d.loc[d["excluir_tch"], "clase_tch_terciles"] = np.nan

# Cortes obtenidos frente a las referencias del EDA y a los umbrales del Ingenio
display(pd.DataFrame({
    "Terciles (train)": [f"{a:.2f} y {b:.2f}" for a, b in cortes_terciles.values()],
    "Terciles de referencia (EDA)": ["133.00 y 153.67", "12.40 y 13.10"],
    "Umbrales del Ingenio": ["125 y 150", "12.2 y 13.0"],
}, index=["TCH", "sacarosa"]))

# Balance de las clases por terciles en ambos conjuntos
balance_terciles = pd.DataFrame({
    ("TCH", "train"): df_train["clase_tch_terciles"].value_counts(normalize=True),
    ("TCH", "test"): df_test["clase_tch_terciles"].value_counts(normalize=True),
    ("sacarosa", "train"): df_train["clase_sac_terciles"].value_counts(normalize=True),
    ("sacarosa", "test"): df_test["clase_sac_terciles"].value_counts(normalize=True),
}).mul(100).round(1)
display(balance_terciles)

print("Registros sin clase de TCH por terciles (train / test):",
      df_train["clase_tch_terciles"].isna().sum(), "/", df_test["clase_tch_terciles"].isna().sum())

,Terciles (train),Terciles de referencia (EDA),Umbrales del Ingenio
TCH,133.00 y 153.00,133.00 y 153.67,125 y 150
sacarosa,12.40 y 13.10,12.40 y 13.10,12.2 y 13.0


TCH       sacarosa      
      train  test    train  test
bajo   34.4  34.0     34.7  35.2
medio  32.3  32.4     32.7  31.5
alto   33.3  33.6     32.6  33.3

Registros sin clase de TCH por terciles (train / test): 4 / 0


### Conclusión de la sección 12

| Target | Terciles (train) | Terciles de referencia (EDA) | Umbrales del Ingenio |
|---|---|---|---|
| TCH | 133 y 153 t/ha | 133 y 153.67 t/ha | 125 y 150 t/ha |
| Sacarosa | 12.4 % y 13.1 % | 12.4 % y 13.1 % | 12.2 % y 13.0 % |

- Los cortes calculados con el conjunto de entrenamiento coinciden con los del dataset completo, salvo una
  diferencia de 0.67 t/ha en el corte superior de TCH. Como TCH se registra en valores enteros, las clases
  resultantes son equivalentes.
- Una vez calculados, los cortes funcionan como umbrales fijos: cada registro de entrenamiento y de prueba
  se clasifica según su propio valor del target. Las clases del conjunto de prueba no dependen de sus
  propios percentiles, igual que las de una cosecha futura no podrían depender de la distribución de
  cosechas que aún no ocurren.
- Las clases por terciles quedan casi balanceadas en ambos conjuntos:

| Clase | TCH train | TCH test | Sacarosa train | Sacarosa test |
|---|---|---|---|---|
| Bajo | 34.4 % | 34.0 % | 34.7 % | 35.2 % |
| Medio | 32.3 % | 32.4 % | 32.7 % | 31.5 % |
| Alto | 33.3 % | 33.6 % | 32.6 % | 33.3 % |

  La proporción en prueba es similar a la de entrenamiento aunque la partición no se estratificó por
  terciles, porque ambos conjuntos provienen de la misma población. Las desviaciones respecto a 33.3 %
  se deben a valores repetidos del target, que deben quedar en la misma clase.
- Durante la validación cruzada, los cortes se mantienen fijos con los valores calculados sobre todo el
  conjunto de entrenamiento, incluidos los pliegues que actúan como validación. Es una aproximación con
  un efecto menor: no altera la evaluación final en el conjunto de prueba.
- Cada target queda con dos definiciones de clase: `clase_tch_ingenio` y `clase_tch_terciles`, y
  `clase_sac_ingenio` y `clase_sac_terciles`.

## 13. Pipeline de preprocesamiento

El preprocesamiento se empaqueta en un pipeline antes de probar cualquier modelo. Así, cada estadístico
se ajusta solo con los datos de entrenamiento de cada pliegue de la validación cruzada, y el conjunto de
prueba recibe exactamente las mismas transformaciones al final.

La sección tiene dos partes: el transformador para `lluvias`, que no existe en scikit-learn y se
construye aparte (13.1), y el pipeline completo con el resto de predictores (13.2).

### 13.1 Transformador para `lluvias`

En el EDA se decidió tratar `lluvias` en dos pasos:

- **Imputación por mes de cosecha.** Los faltantes se reemplazan con la mediana de `lluvias` de su mes,
  porque la lluvia tiene un patrón estacional. Si un mes no tiene datos observados en el ajuste, se usa
  la mediana general.
- **Acotamiento al percentil 99.** Los valores por encima del percentil 99 se reemplazan por ese valor,
  para que unos pocos registros extremos no dominen las distancias de KNN.

Las medianas y el tope se calculan solo con los valores observados del conjunto de ajuste. El
transformador recibe `lluvias` y `mes`, y devuelve solo `lluvias`: el mes se usa para elegir la mediana,
pero no entra al modelo por esta vía, porque ya está representado por sus armónicos.

Antes de incluirlo en el pipeline, se prueba por separado: se ajusta con el conjunto de entrenamiento y
se verifica que no deja faltantes, que ningún valor supera el tope y que los valores observados por
debajo del tope no cambian.

In [7]:
# 13.1 Transformador para lluvias: mediana por mes de cosecha y acotamiento al percentil 99
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted


class ImputadorLluvias(BaseEstimator, TransformerMixin):
    """Imputa lluvias con la mediana de su mes de cosecha y acota los valores al percentil indicado.

    Recibe dos columnas en este orden: lluvias y mes. Devuelve solo lluvias.
    Las medianas y el tope se calculan con los valores observados del conjunto de ajuste.
    """

    def __init__(self, percentil_tope=99):
        self.percentil_tope = percentil_tope

    def fit(self, X, y=None):
        lluvias, mes = self._separar(X)
        observado = ~np.isnan(lluvias)
        self.mediana_mes_ = pd.Series(lluvias[observado]).groupby(mes[observado]).median().to_dict()
        self.mediana_global_ = float(np.median(lluvias[observado]))
        self.tope_ = float(np.percentile(lluvias[observado], self.percentil_tope))
        self.n_features_in_ = 2
        return self

    def transform(self, X):
        check_is_fitted(self, "tope_")
        lluvias, mes = self._separar(X)
        # Mediana del mes; si el mes no tuvo datos en el ajuste, mediana general
        mediana = pd.Series(mes).map(self.mediana_mes_).fillna(self.mediana_global_).to_numpy()
        lluvias = np.where(np.isnan(lluvias), mediana, lluvias)
        return np.minimum(lluvias, self.tope_).reshape(-1, 1)

    def get_feature_names_out(self, input_features=None):
        return np.array(["lluvias"], dtype=object)

    @staticmethod
    def _separar(X):
        X = np.asarray(X, dtype=float)
        return X[:, 0], X[:, 1].astype(int)


# Prueba aislada: ajuste solo con el conjunto de entrenamiento
imputador = ImputadorLluvias().fit(df_train[["lluvias", "mes"]])
lluvias_train = imputador.transform(df_train[["lluvias", "mes"]]).ravel()
lluvias_test = imputador.transform(df_test[["lluvias", "mes"]]).ravel()

assert not np.isnan(lluvias_train).any() and not np.isnan(lluvias_test).any(), "Quedaron faltantes"
assert max(lluvias_train.max(), lluvias_test.max()) <= imputador.tope_, "Hay valores sobre el tope"
sin_cambio = (df_train["lluvias"] <= imputador.tope_).to_numpy()  # NaN da False
assert np.array_equal(lluvias_train[sin_cambio], df_train["lluvias"].to_numpy()[sin_cambio]), \
    "Cambiaron valores observados por debajo del tope"

print(f"Mediana general: {imputador.mediana_global_:.1f} | Tope (P{imputador.percentil_tope}): "
      f"{imputador.tope_:.1f}")

# Mediana usada para imputar en cada mes de cosecha
display(pd.DataFrame({
    "Registros (train)": df_train.groupby("mes").size(),
    "Con dato (train)": df_train.groupby("mes")["lluvias"].count(),
    "Mediana imputada": pd.Series(imputador.mediana_mes_).round(1),
}).rename_axis("mes"))

# Registros afectados en cada conjunto
display(pd.DataFrame({
    "Imputados": [df_train["lluvias"].isna().sum(), df_test["lluvias"].isna().sum()],
    "Acotados al tope": [(df_train["lluvias"] > imputador.tope_).sum(),
                         (df_test["lluvias"] > imputador.tope_).sum()],
}, index=["train", "test"]))

Mediana general: 130.0 | Tope (P99): 720.3


,Registros (train),Con dato (train),Mediana imputada
mes,,,
1,165,96,134.0
2,63,50,91.5
3,153,106,100.0
4,151,110,177.0
5,141,79,177.0
6,176,124,189.0
7,105,65,160.0
8,131,96,119.0
9,180,123,74.0


,Imputados,Acotados al tope
train,555,12
test,130,1


### Conclusión de 13.1

- El transformador cumple las tres verificaciones: no deja faltantes, ningún valor supera el tope y los
  valores observados por debajo del tope no cambian.
- Se imputan 555 registros de entrenamiento (31.7 %) y 130 de prueba (29.7 %), que suman los 685
  faltantes identificados en el EDA.
- Los 12 meses tienen datos observados en el conjunto de entrenamiento, con al menos 50 registros por
  mes (febrero es el de menos), así que todas las imputaciones usan la mediana de su mes y ninguna la
  mediana general.
- Las medianas por mes van de 74 mm (septiembre) a 206 mm (diciembre), frente a una mediana general de
  130 mm. Imputar con la mediana general habría asignado el mismo valor a cosechas de meses con lluvias
  muy distintas.
- El tope es 720.3 mm, más de cinco veces la mediana. Se acotan 12 registros de entrenamiento y 1 de
  prueba, así que el acotamiento solo actúa sobre los valores más extremos.
- Los valores imputados son una aproximación, no una medición. La bandera `lluvia_sin_dato` permite al
  modelo distinguir los registros con lluvia medida de los imputados.

### 13.2 Pipeline completo

El pipeline combina el transformador de `lluvias` con el tratamiento del resto de predictores:

| Grupo | Columnas | Transformación |
|---|---|---|
| Lluvias | `lluvias` (con `mes` como apoyo) | Mediana por mes y acotamiento al percentil 99 (13.1) |
| Numéricas | `edad`, `cortes`, `semsmad`, `pct_diatrea`, `dosismad`, `anio` | Imputación con la mediana |
| Directas | `lluvia_sin_dato` y los cuatro armónicos del mes | Ninguna antes del escalado |
| Categórica | `grupo_tenencia` | Indicadoras, con el código 1 como referencia |

Después, todas las columnas se estandarizan. KNN mide distancias, así que sin escalado dominarían las
variables con valores más grandes, como `lluvias` o `anio`. En la regresión logística, el escalado hace que
la regularización penalice por igual a todos los coeficientes y que sus magnitudes sean comparables.

Las variables binarias y las indicadoras también se escalan, para que tengan el mismo peso que las
continuas en las distancias de KNN y en la penalización de la logística. Solo se imputa con la mediana
en `semsmad` y `dosismad`, que tienen 2 faltantes cada una. En el resto de columnas numéricas el
imputador no cambia ningún valor.

El pipeline se construye con una función, porque en la sección 16 se construye con otros conjuntos de
predictores. Se ajusta con el conjunto de entrenamiento y se verifica que no quedan faltantes, que `mes`
no llega al modelo y que las columnas de entrenamiento tienen media 0 y desviación 1. Por último, se
calcula el VIF sobre la matriz de diseño de entrenamiento, que ahora incluye `lluvias` imputada, la
bandera, los armónicos y las indicadoras.

In [8]:
# 13.2 Pipeline de preprocesamiento
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# mes solo entra para elegir la mediana de lluvias; el ColumnTransformer no lo devuelve
columnas_entrada = predictores + ["mes"]
predictores_num_mediana = [c for c in predictores_num if c != "lluvias"]


def crear_preprocesador(num=predictores_num_mediana, directas=predictores_bin + predictores_mes,
                        cat=predictores_cat):
    """Imputación, codificación y escalado. Todo se ajusta con los datos que recibe fit()."""
    columnas = ColumnTransformer(
        [
            ("lluvias", ImputadorLluvias(), ["lluvias", "mes"]),
            ("num", SimpleImputer(strategy="median"), num),
            ("directas", "passthrough", directas),
            ("cat", OneHotEncoder(drop="first", sparse_output=False), cat),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )
    return Pipeline([("columnas", columnas), ("escalado", StandardScaler())]).set_output(transform="pandas")


preprocesador = crear_preprocesador()
X_train_prep = preprocesador.fit_transform(df_train[columnas_entrada])
X_test_prep = preprocesador.transform(df_test[columnas_entrada])

assert X_train_prep.notna().all().all() and X_test_prep.notna().all().all(), "Quedaron faltantes"
assert "mes" not in X_train_prep.columns, "mes no debe llegar al modelo"
assert np.allclose(X_train_prep.mean(), 0) and np.allclose(X_train_prep.std(ddof=0), 1), \
    "El escalado no se ajustó con train"

print("Matriz de diseño (train):", X_train_prep.shape, "| (test):", X_test_prep.shape)

# En test, media y desviación cercanas a 0 y 1 pero no exactas, porque el escalado viene de train
display(pd.DataFrame({
    "Media (test)": X_test_prep.mean(),
    "Desv. estándar (test)": X_test_prep.std(ddof=0),
}).round(2).T)

Matriz de diseño (train): (1749, 14) | (test): (438, 14)


,lluvias,edad,cortes,semsmad,pct_diatrea,dosismad,anio,lluvia_sin_dato,mes_sin_12,mes_cos_12,mes_sin_6,mes_cos_6,grupo_tenencia_2,grupo_tenencia_3
Media (test),-0.02,-0.01,0.08,-0.01,0.03,0.04,-0.01,-0.04,-0.01,0.18,-0.04,0.13,0.04,-0.02
Desv. estándar (test),0.99,1.06,1.09,1.07,0.91,1.06,1.01,0.98,0.95,1.03,1.00,0.99,1.01,1.00


In [9]:
# 13.2 VIF sobre la matriz de diseño de entrenamiento
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_vif = sm.add_constant(X_train_prep)
vif_diseno = pd.Series(
    [variance_inflation_factor(X_vif.to_numpy(), i) for i in range(1, X_vif.shape[1])],
    index=X_train_prep.columns, name="VIF",
).sort_values(ascending=False).round(2)
display(vif_diseno.to_frame())

,VIF
grupo_tenencia_3,3.38
edad,2.53
grupo_tenencia_2,2.49
semsmad,2.22
lluvias,2.04
lluvia_sin_dato,1.52
dosismad,1.50
anio,1.48
cortes,1.28
mes_cos_6,1.16


### Conclusión de 13.2

- La matriz de diseño tiene 14 columnas: 7 numéricas, la bandera de lluvia, los 4 armónicos del mes y 2
  indicadoras de `grupo_tenencia`. Tiene 1.749 filas en entrenamiento y 438 en prueba, sin faltantes.
- En prueba, las medias están entre −0.04 y 0.18 y las desviaciones entre 0.91 y 1.09. No son exactamente
  0 y 1 porque el escalado se ajustó con el conjunto de entrenamiento. Las mayores diferencias están en
  `mes_cos_12` (0.18) y `mes_cos_6` (0.13): el conjunto de prueba tiene una distribución de meses de
  cosecha algo distinta, porque la partición se estratificó por clase y no por mes.
- Todos los VIF están por debajo de 5:

| Predictor | VIF |
|---|---|
| `grupo_tenencia_3` | 3.38 |
| `edad` | 2.53 |
| `grupo_tenencia_2` | 2.49 |
| `semsmad` | 2.22 |
| `lluvias` | 2.04 |
| `lluvia_sin_dato` | 1.52 |
| `dosismad` | 1.50 |
| `anio` | 1.48 |
| `cortes` | 1.28 |
| Armónicos del mes | 1.06 a 1.16 |
| `pct_diatrea` | 1.07 |

- Los VIF más altos corresponden a las indicadoras de `grupo_tenencia`. Con el código 1 como referencia,
  que solo tiene el 12.5 % de los registros, casi toda cosecha que no es del código 2 es del código 3, así
  que las dos indicadoras están fuertemente relacionadas entre sí. Es una consecuencia de la codificación y
  no de un solapamiento con otros predictores.
- `lluvias` tiene un VIF de 2.04. En el 31.7 % de los registros su valor es la mediana de su mes, así que
  queda relacionada con los armónicos y con la bandera, pero no al punto de afectar la interpretación de
  los coeficientes.

### Conclusión de la sección 13

El preprocesamiento queda empaquetado en un pipeline que imputa `lluvias` por mes y la acota al
percentil 99, imputa con la mediana `semsmad` y `dosismad`, codifica `grupo_tenencia` y estandariza todas
las columnas. Todos sus estadísticos se ajustan con los datos que recibe en el entrenamiento, de modo que
en la validación cruzada se recalculan en cada pliegue. Los predictores no presentan multicolinealidad
relevante, así que los coeficientes de la regresión logística se pueden interpretar individualmente.

## 14. Regresión logística multinomial

La regresión logística multinomial estima, para cada registro, la probabilidad de pertenecer a cada una
de las tres clases, y asigna la más probable. Es el modelo base del enunciado: sus coeficientes indican
qué predictores empujan una cosecha hacia la clase baja o hacia la alta. Con regularización L1, algunos
coeficientes se anulan, lo que sirve como selección de variables.

El modelo se ajusta en cuatro casos: dos targets (TCH y sacarosa) por dos definiciones de clase (umbrales
del Ingenio y terciles). La sección tiene tres partes: el esquema de validación cruzada y los modelos de
referencia (14.1), la búsqueda de hiperparámetros (14.2) y el análisis de los coeficientes (14.3).

### 14.1 Esquema de validación cruzada y modelos de referencia

Los hiperparámetros se eligen con validación cruzada de 5 pliegues sobre el conjunto de entrenamiento.
Los pliegues se construyen igual que la partición de la sección 11: agrupados por suerte, para que el
desempeño en validación corresponda a suertes no vistas, y estratificados por la clase de cada caso. El
pipeline de la sección 13 se ajusta de nuevo en cada pliegue.

En cada pliegue se calculan cuatro métricas:

- **Exactitud:** proporción de registros bien clasificados.
- **F1 macro:** promedio simple del F1 de las tres clases. Da el mismo peso a cada clase aunque tengan
  tamaños distintos, así que es la métrica principal para elegir hiperparámetros.
- **Sensibilidad de la clase baja:** proporción de cosechas de desempeño bajo que el modelo detecta.
- **Kappa de Cohen:** acuerdo entre las clases predichas y las reales, descontando el acuerdo esperado
  por azar. Vale 0 para un clasificador aleatorio y 1 para uno perfecto.

Para saber si un modelo aprende algo, se comparan sus métricas con dos referencias que no usan los
predictores: un clasificador que siempre predice la clase más frecuente y uno que asigna clases al azar
con las proporciones de entrenamiento. Un modelo útil debe superarlas con claridad en F1 macro, en
sensibilidad de la clase baja y en kappa.